In [0]:
#Types of Window Functions:
#1) Analytical Function: e.g., lead(), lag(), cume_dist()
#2) Ranking Function: e.g., row_number(), rank(), dense_rank(), percent_rank()
#3) Aggregate Function: e.g., avg(), sum(), min(), max()

data = [(1,"ramesh",'it',1000),
        (2,"suresh",'hr',2000),
        (3,"raju",'it',3000),
        (4,"rajesh",'hr',4000),
        (5,"ravi",'hr',5000)        
        ]
column = [ 'id','name','dept','salary']

from pyspark.sql.functions import col,dense_rank,rank,row_number,lag,lead
from pyspark.sql.window import Window
df = spark.createDataFrame(data, column)
df.show()

print("dense_rank")
WindowSpec = Window.orderBy(col('salary').desc())
df_with_moving_avg = df.withColumn('DenseRank',dense_rank().over(WindowSpec))
#second_highest_salary = df_with_moving_avg.filter(col('DenseRank') == 2).select('salary').distinct()
#second_highest_salary.show()
df_with_moving_avg.show()

print("rank")
windowSpec = Window.partitionBy('dept').orderBy(col('salary').desc())
df_with_rank = df.withColumn('rank',rank().over(windowSpec))
df_with_rank.show()

print("row_number")
windowSpec = Window.partitionBy('dept').orderBy(col('salary').desc())
df_with_row = df.withColumn('rowrank',row_number().over(windowSpec))
df_with_row.show()

print("lag")
windowSpec = Window.partitionBy('dept').orderBy(col('salary').desc())
df_with_lag = df.withColumn('lag',lag(col('salary'),1).over(windowSpec))
df_with_lag.show()

print("lead")
window_spec = Window.partitionBy("dept").orderBy("salary")
df_lead = df.withColumn("next_salary", lead("salary", 1).over(window_spec))
df_lead.show()
df.show()

In [0]:
from pyspark.sql.functions import col, dense_rank

# Find the second highest distinct salary
df_with_rank = df.withColumn(
    "rank",
    dense_rank().over(Window.orderBy(col("salary").desc()))
)
second_highest_salary = df_with_rank.filter(col("rank") == 2).select("salary").distinct()
display(second_highest_salary)






raw data(Key, Value): text, your name, number, 1$2#Z34A!, role, Bank Employee
 
1. create a file for the above raw data

2. read the data-file as Key/Value pair. And extract the value(number) from the string.

3. reverse the value using any of OOPs concepts, if Key is "number"

4. display the reversed number in JSON data  { reversed number, 4321 } & store it in a json file.
 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, reverse

# 🔹 Create Spark Session
spark = SparkSession.builder.appName("KeyValueProcessing").getOrCreate()

# 🔹 Step 1: Read file
df = spark.read.text("data.txt")

# 🔹 Step 2: Split into key and value
df_kv = df.withColumn("key", col("value").substr(1, instr(col("value"), "=")-1)) \
          .withColumn("val", col("value").substr(instr(col("value"), "=")+1, 100))

# 🔹 Step 3: Filter 'number' key
number_df = df_kv.filter(col("key") == "number")

# 🔹 Step 4: Extract only digits
digits_df = number_df.withColumn(
    "digits",
    regexp_replace(col("val"), "[^0-9]", "")
)

# 🔹 Step 5: Reverse digits
result_df = digits_df.withColumn(
    "reversed_number",
    reverse(col("digits"))
)

# 🔹 Step 6: Select final output
final_df = result_df.select("reversed_number")

# Show result
final_df.show()

# 🔹 Step 7: Write to JSON
final_df.write.mode("overwrite").json("output_json")

